In [7]:
from flask import Flask, jsonify
from pydicom.dataset import Dataset
from pynetdicom import AE
from pynetdicom.sop_class import ModalityWorklistInformationFind

app = Flask(__name__)

@app.route('/worklist', methods=['GET'])
def get_worklist():
    # Configura la AE y el servidor DCM4CHEE
    ae = AE(ae_title='MY_AE_TITLE')  # Configura tu AE Title
    ae.add_requested_context(ModalityWorklistInformationFind)

    # Dirección y puerto del servidor DCM4CHEE
    server_ip = '192.168.31.56'
    server_port = 2575

    # Define el dataset de consulta
    ds = Dataset()
    ds.PatientName = '*'  # Consulta todos los pacientes
    ds.QueryRetrieveLevel = 'PATIENT'  # Asegúrate de establecer el nivel de consulta adecuado

    # Realiza la consulta C-FIND
    assoc = ae.associate(server_ip, server_port)
    if assoc.is_established:
        responses = assoc.send_c_find(ds, query_model=ModalityWorklistInformationFind)
        worklist = []
        for (status, identifier) in responses:
            if status and identifier:
                # Convierte el Dataset a un diccionario para JSON
                worklist.append({elem.tag: elem.value for elem in identifier})
        assoc.release()
        return jsonify(worklist)
    else:
        return jsonify({"error": "No se pudo establecer conexión con el servidor DCM4CHEE"}), 500

if __name__ == '__main__':
    app.run(debug=True)


 * Tip: There are .env or .flaskenv files present. Do "pip install python-dotenv" to use them.


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\Users\Farias - Medical IT\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3556: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [8]:
%tb

SystemExit: 1

In [15]:
import logging
from pydicom.dataset import Dataset
from pynetdicom import AE, debug_logger
from pynetdicom.sop_class import ModalityWorklistInformationFind

# Habilitar el registro de depuración
debug_logger()

ae = AE(ae_title='TEST1')  # Configura tu AE Title
ae.add_requested_context(ModalityWorklistInformationFind)

# Dirección y puerto del servidor DCM4CHEE
server_ip = '192.168.31.56'
server_port = 2575

# Define el dataset de consulta
ds = Dataset()
ds.PatientName = '*'  # Consulta todos los pacientes
ds.QueryRetrieveLevel = 'PATIENT'  # Asegúrate de establecer el nivel de consulta adecuado

# Realiza la consulta C-FIND
assoc = ae.associate(server_ip, server_port, ae_title='WORKLIST')
if assoc.is_established:
    responses = assoc.send_c_find(ds, query_model=ModalityWorklistInformationFind)
    worklist = []
    for (status, identifier) in responses:
        if status and identifier:
            # Convierte el Dataset a un diccionario para JSON
            worklist.append({elem.tag: elem.value for elem in identifier})
    print(worklist)
    assoc.release()
else:
    print("No se pudo establecer conexión con el servidor DCM4CHEE")




I: Requesting Association
D: Request Parameters:
D: ======================= OUTGOING A-ASSOCIATE-RQ PDU ========================
D: Our Implementation Class UID:      1.2.826.0.1.3680043.9.3811.2.1.0
D: Our Implementation Version Name:   PYNETDICOM_210
D: Application Context Name:    1.2.840.10008.3.1.1.1
D: Calling Application Name:    TEST1
D: Called Application Name:     WORKLIST
D: Our Max PDU Receive Size:    16382
D: Presentation Context:
D:   Context ID:        1 (Proposed)
D:     Abstract Syntax: =Modality Worklist Information Model - FIND
D:     Proposed SCP/SCU Role: Default
D:     Proposed Transfer Syntaxes:
D:       =Implicit VR Little Endian
D:       =Explicit VR Little Endian
D:       =Deflated Explicit VR Little Endian
D:       =Explicit VR Big Endian
D: Requested Extended Negotiation: None
D: Requested Common Extended Negotiation: None
D: Requested Asynchronous Operations Window Negotiation: None
D: Requested User Identity Negotiation: None
D: ==========================

No se pudo establecer conexión con el servidor DCM4CHEE
